In [1]:
import pandas as pd
import numpy as np
from NoisyCircuits.utils.CreateNoiseModel import CreateNoiseModel
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError

2026-02-24 17:58:49,319	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
import pickle

with open("../noise_models/Noise_Model_Heron_QPU.pkl", "rb") as f:
    noise_model = pickle.load(f)

In [2]:
df = pd.read_csv("../noise_models/sample_data.csv", delimiter=",")

In [2]:
model = CreateNoiseModel(calibration_data_file="../noise_models/sample_data.csv",
                         basis_gates=[["x", "sx", "rx"], ["cz", "rzz"]])
noise_model_from_csv = model.create_noise_model()

No Error Data found for gate cz on qubit 72
Attempting to apply error data from other two qubit gates for qubit 72
No Error Data found for gate rzz on qubit 99
Attempting to apply error data from other two qubit gates for qubit 99


In [38]:
from NoisyCircuits.utils.BuildQubitGateModel import BuildModel
from pympler import asizeof

In [47]:
model = BuildModel(noise_model=noise_model_from_csv,
                   num_qubits=20,
                   num_cores=20,
                   threshold=1e-4,
                   basis_gates=[["x", "sx", "rx", "rz"], ["cz", "rzz"]],
                   verbose=False).build_qubit_gate_model()

In [53]:
print("Size of noise model: ", asizeof.asizeof(model) / 1024 / 1024 / 1024, "GB")

Size of noise model:  96.81760073453188 GB


In [1]:
import qiskit_ibm_runtime
import json
import os

In [2]:
token = json.load(open(os.path.join(os.path.expanduser("~"), "ibm_api.json"), "r"))["apikey"]

In [3]:
service = qiskit_ibm_runtime.QiskitRuntimeService(channel="ibm_quantum_platform", token=token)

qiskit_runtime_service._discover_account:WARNING:2026-02-27 17:13:26,975: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-02-27 17:13:31,410: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Open_Sys. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


In [4]:
dev = service.backend("ibm_fez")

qiskit_runtime_service.backends:WARNING:2026-02-27 17:13:34,172: Using instance: Open_Sys, plan: open


In [7]:
dev.basis_gates

['cz', 'id', 'rz', 'sx', 'x']

In [24]:
dev.target["cz"][(0, 1)]

InstructionProperties(duration=6.8e-08, error=0.011462126371348552)

In [29]:
dev.target["rzz"][(0,1)]

KeyError: 'rzz'

In [25]:
# Extract qubit-specific data
props = dev.properties()
qubit_data = []
for i, qubit in enumerate(props.qubits):
    qubit_info = {"qubit": i}
    for item in qubit:
        qubit_info[item.name] = item.value
    qubit_data.append(qubit_info)

# Create a DataFrame and export
df = pd.DataFrame(qubit_data)
# df.to_csv("calibration_data.csv", index=False)

In [27]:
df.head(1)

,qubit,T1,T2,readout_error,prob_meas0_prep1,prob_meas1_prep0,readout_length
0,0,48.758461,16.172514,0.020996,0.040527,0.001465,1560
